# 전북도 고속도로 결빙 분석: 코드와 결과 확인

2026-09-23에 실행한 **데이터 감사 → 공간·시각 점검 → 개발용 EDA → 관측소 단위 검증 → 첫 ASOS 기준 실험**을 한곳에서 확인한다. 공식 보호 RWIS 3개 지점(12410·33411·23402)의 원자료를 다시 열지 않는다. 이미 완료된 고정 평가 기록과 개발 결과를 구분해 읽는다.

노트북에는 원본 자료와 예측 행 출력이 저장돼 있지 않다. 아래 셀은 로컬의 Git 제외 경로 `outputs/private/`에 있는 집계 결과를 읽는다. 결과가 없으면 '전체 재생성' 셀의 스위치를 켜서 다시 만든다. 10분 결빙 평가 방식과 필수 추정 변수는 운영진 확인 전이므로 현재 점수는 **개발용 잠정 결과**다.

Update: Development-only 10-minute Ice predictions, provisional daily metrics, paired AWS-input comparison, and one-slot event context are now included below. The original protected assessment has already been opened once; review reads frozen aggregate receipts without reassessment.

## 2026-10-07 수정본 안내
루트 기본 원본·캐시는 20260915로 보존되어 있다. 수정본 분석은 `outputs/private/releases/20261007/`의 작업 복사본에서 실행한다. 두 환경의 캐시를 섞지 않는다. 최신 Q&A·재실행 결과는 [10월 7일 보고서](../docs/competition/OFFICIAL_UPDATE_20261007.md)를 확인한다. 복사본에서는 재생성한 단계만 수정본 결과가 있으며 과거 HGB·공간 분석 산출물을 자동으로 가져오지 않는다.


In [ ]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "config" / "evaluation.json").exists() and (p / "scripts" / "audit_data.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("프로젝트 폴더 또는 notebooks/ 폴더에서 이 노트북을 열어주세요.")
PRIVATE = ROOT / "outputs" / "private"
CONFIG = json.loads((ROOT / "config" / "evaluation.json").read_text(encoding="utf-8"))
PROTECTED = set(CONFIG["held_out_rwis_stations"])
print("프로젝트:", ROOT)
print("Python:", platform.python_version(), "| 보호 RWIS:", ", ".join(sorted(PROTECTED)))

def read_json(relative):
    path = PRIVATE / relative
    if not path.exists():
        raise FileNotFoundError(f"{path} 없음. 아래 전체 재생성 셀을 실행하세요.")
    return json.loads(path.read_text(encoding="utf-8"))

SOURCE_CONFIG = json.loads((ROOT / "config/data_source.json").read_text(encoding="utf-8"))
print("원본 배포본:", SOURCE_CONFIG["release"], "| SHA256:", SOURCE_CONFIG["sha256"])
SKIPPED_REVIEW_CELLS = []


## 실행 코드

| 단계 | 스크립트 | 확인 내용 |
| --- | --- | --- |
| 파일·변수 감사 | [audit_data.py](../scripts/audit_data.py) | 배포본별 720/960개 CSV, 관측소·품질 |
| 날짜 경계 | [audit_timestamps.py](../scripts/audit_timestamps.py) | 빈 행, 자정 중복 |
| AWS 자정 중복 | [audit_aws_boundaries.py](../scripts/audit_aws_boundaries.py) | 수치 표기 동등성과 값 충돌 |
| AWS 입력 필터 | [audit_aws_cleaning.py](../scripts/audit_aws_cleaning.py), [weather_inputs.py](../src/weather_inputs.py) | 파일 날짜·KST·제외 건수 |
| 공간자료 | [audit_spatial.py](../scripts/audit_spatial.py) | 중심선·DEM·중첩 레이어 |
| 도로 위치 검증 | [audit_station_road_alignment.py](../scripts/audit_station_road_alignment.py) | 개발 RWIS·노선 중심선 거리 |
| 변수 사전 | [make_variable_dictionary.py](../scripts/make_variable_dictionary.py) | 실제 열·단위·형 |
| 개발용 EDA | [eda_development.py](../scripts/eda_development.py) | 보호 지점 제외 결빙 분포 |
| 분할 | [make_splits.py](../scripts/make_splits.py) | 외부 LOSO 14회·내부 3분할 |
| 첫 기준 모델 | [baseline_asos.py](../scripts/baseline_asos.py) | 과거 최근접 ASOS로 노면온도 추정 |
| 기상 특징 | [build_weather_features.py](../scripts/build_weather_features.py), [weather_features.py](../src/weather_features.py) | 과거 AWS·ASOS 시각 결합·연결률 |
| 기상 모델 | [baseline_weather_ridge.py](../scripts/baseline_weather_ridge.py) | 기상 융합 Ridge LOSO 점수 |
| 실패 진단 | [diagnose_weather_ridge.py](../scripts/diagnose_weather_ridge.py) | 겨울·AWS 연결 여부별 OOF 오차 |
| 시간 일반화 | [evaluate_weather_temporal.py](../scripts/evaluate_weather_temporal.py) | 첫 겨울 학습·둘째 겨울 보조 검증 |
| 평가 함수·테스트 | [validation.py](../src/validation.py), [test_validation.py](../tests/test_validation.py) | BIAS, 일별 지표 후보, 보호 지점 |

다음 셀은 테스트와 이틀짜리 작은 실행만 다시 수행한다. 전체 파일의 감사나 240일 기준 모델은 자동 재실행하지 않는다.

In [ ]:
def captured_lines(stdout, stderr):
    text = "\n".join(part for part in (stdout, stderr) if part)
    return [line for line in text.splitlines() if line.strip()]


assert captured_lines(None, "ok") == ["ok"]
assert captured_lines("ok", None) == ["ok"]
assert captured_lines(None, None) == []

SMOKE_COMMANDS = [
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    [sys.executable, "scripts/audit_data.py", "--limit-files", "3"],
    [sys.executable, "scripts/baseline_asos.py", "--limit-days", "2"],
]
# 예측·모델이 고정된 작업 공간에서는 검토용 읽기만 수행한다.
frozen_receipts = [
    PRIVATE / "external_reserved_predictions_v1/predictions_receipt.json",
    PRIVATE / "releases/20261007/outputs/private/external_reserved_predictions_v1/predictions_receipt.json",
]
if any(path.exists() for path in frozen_receipts):
    print("예측 고정 영수증 확인: 테스트·소규모 산출물 재생성은 건너뛰고 기존 결과를 검토합니다.")
else:
    for command in SMOKE_COMMANDS:
        completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, check=True)
        lines = captured_lines(completed.stdout, completed.stderr)
        print("OK", " ".join(command[1:]), " | ", lines[-1] if lines else "성공")


## 단계별 실행과 산출물 확인

아래 셀의 `RUN_STAGES`에 [실행 목록](../config/notebook_workflow.json)의 단계 ID를 넣고 실행한다. 예를 들어 `['baseline_asos']`는 ASOS 기준 실험만 다시 실행한다. `REBUILD_FULL=True`는 등록된 전체 단계를 순서대로 실행한다. 기본값은 재실행 없이 현재 로컬 산출물의 존재·수정 시각·선별 집계값을 확인한다. 실행 뒤 아래 결과 셀을 다시 실행하면 표와 그림이 갱신된다.

앞으로 새 단계는 스크립트와 요약 산출물을 만든 뒤 실행 목록에 등록하고, 필요한 해석 셀을 이 노트북에 추가한다. 제공 데이터 원본 행이나 OOF 예측 원자료는 노트북 출력으로 저장하지 않는다.

In [ ]:
WORKFLOW = json.loads((ROOT / "config" / "notebook_workflow.json").read_text(encoding="utf-8"))
assert WORKFLOW["schema_version"] == 1
STAGES = {stage["id"]: stage for stage in WORKFLOW["stages"]}
assert len(STAGES) == len(WORKFLOW["stages"])

# 필요한 단계만 선택해 실행한다. 전체 원본 스캔은 오래 걸리므로 기본값은 실행하지 않는다.
RUN_STAGES = []  # 예: ["weather_features", "weather_ridge"]
if os.environ.get("ICY_ROAD_RUN_STAGES"):
    RUN_STAGES = json.loads(os.environ["ICY_ROAD_RUN_STAGES"])
REBUILD_FULL = False
unknown = set(RUN_STAGES) - set(STAGES)
if unknown:
    raise ValueError(f"등록되지 않은 단계: {sorted(unknown)}")
eligible = [key for key, stage in STAGES.items() if SOURCE_CONFIG["release"] in stage.get("releases", [SOURCE_CONFIG["release"]])]
selected = eligible if REBUILD_FULL else RUN_STAGES
if set(selected) - set(eligible):
    raise ValueError("현재 배포본에서 제공되지 않는 단계입니다. 수정본 작업 폴더를 사용하세요.")
for stage_id in selected:
    stage = STAGES[stage_id]
    command = [sys.executable, *stage["command"]]
    completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, check=True)
    artifact = ROOT / stage["artifact"]
    if not artifact.exists():
        raise FileNotFoundError(f"{stage_id} 실행 후 산출물이 없습니다: {artifact}")
    print("OK", stage_id, "|", stage["artifact"])

status_rows = []
summary_rows = []
for stage in WORKFLOW["stages"]:
    artifact = ROOT / stage["artifact"]
    status_rows.append({
        "단계 ID": stage["id"],
        "설명": stage["name"],
        "산출물": stage["artifact"],
        "존재": artifact.exists(),
        "수정 시각 (KST)": pd.Timestamp(artifact.stat().st_mtime, unit="s", tz="UTC")
            .tz_convert("Asia/Seoul").strftime("%Y-%m-%d %H:%M") if artifact.exists() else None,
    })
    if artifact.suffix == ".json" and artifact.exists():
        result = json.loads(artifact.read_text(encoding="utf-8"))
        for key in stage.get("summary_keys", []):
            if key in result:
                value = result[key]
                summary_rows.append({"단계 ID": stage["id"], "지표": key,
                                     "값": json.dumps(value, ensure_ascii=False)})
display(pd.DataFrame(status_rows))
display(pd.DataFrame(summary_rows))


## 1. 파일·시각 품질

자료군별 행 수와 실측 지점을 확인한다. 빈 ID 행은 실측 지점 수에서 뺀다. AWS 자정 중복의 '원시 문자열 차이'는 숫자 값 차이로 확정되지 않았다.

In [ ]:
required_artifacts = ["full/audit_summary.json", "full/timestamp_summary.json"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(8)
    print("미생성 단계: 셀 8 |", ", ".join(missing_artifacts))
else:
    audit = read_json("full/audit_summary.json")
    timing = read_json("full/timestamp_summary.json")
    audit_rows = []
    for kind in ("RWIS", "ASOS", "AWS"):
        ts = timing[kind]
        audit_rows.append({
            "자료": kind,
            "일별 파일": audit["file_counts"][kind],
            "행": audit["row_counts"][kind],
            "실측 지점": audit["station_counts"][kind] - int(ts["missing_id_rows"] > 0),
            "완전 빈 행": ts["empty_placeholder_rows"],
            "다음 날 00:00 행": ts["next_midnight_rows"],
            "다음 파일과 중복": ts["cross_file_midnight"].get("matched", 0),
        })
    display(pd.DataFrame(audit_rows))
    assert {"RWIS", "ASOS", "AWS"} <= set(audit["file_counts"]) <= {"RWIS", "ASOS", "AWS", "FOG"}
    assert all(count == 240 for count in audit["file_counts"].values())
    if "FOG" in audit["file_counts"]:
        print("추가 안개 자료:", audit["file_counts"]["FOG"], "파일 /", audit["row_counts"]["FOG"], "행 /", audit["station_counts"]["FOG"], "지점")
    assert timing["AWS"]["empty_placeholder_rows"] == timing["AWS"]["missing_id_rows"]


### AWS 자정 중복과 입력 날짜 필터

같은 지점·같은 시각의 자정 행은 원시 문자열과 수치 정규화 값을 따로 비교했다. 원본 ZIP은 수정하지 않았다. AWS 모델 입력은 파일명 날짜의 KST 행만 유지한다. 이 단계는 관측값을 보간하거나 이상한 측정치를 복원하지 않는다.

In [ ]:
required_artifacts = ["full/audit_summary.json", "full/aws_boundary_audit.json", "full/aws_cleaning_summary.json", "full/timestamp_summary.json"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(10)
    print("미생성 단계: 셀 10 |", ", ".join(missing_artifacts))
else:
    boundary = read_json("full/aws_boundary_audit.json")
    cleaning = read_json("full/aws_cleaning_summary.json")
    b = boundary["counts"]
    c = cleaning["counts"]
    display(pd.DataFrame([
        {"점검": "AWS 자정 중복", "수치": b.get("matched", 0)},
        {"점검": "수치 정규화 후 동일", "수치": b.get("equivalent_after_numeric_normalization", 0)},
        {"점검": "원시 문자열 동일", "수치": b.get("raw_identical", 0)},
        {"점검": "지점 없는 행 제외", "수치": c["missing_station_rows"]},
        {"점검": "파일명 날짜 밖 행 제외", "수치": c["valid_station_outside_nominal_day"]},
        {"점검": "유지한 AWS 행", "수치": c["retained_rows"]},
    ]))
    print("짝이 없는 자정 행의 날짜:", boundary["unmatched_midnight_dates"])
    assert b.get("matched", 0) == b.get("equivalent_after_numeric_normalization", 0)
    assert b.get("matched", 0) == timing["AWS"]["cross_file_midnight"].get("matched", 0)
    assert not boundary["changed_columns"]
    assert c["input_rows"] == audit["row_counts"]["AWS"]
    assert c["missing_station_rows"] == timing["AWS"]["missing_id_rows"]
    assert c["valid_station_outside_nominal_day"] == timing["AWS"]["next_midnight_rows"]
    assert c["duplicate_station_time_rows"] == 0
    assert c["input_rows"] == (c["missing_station_rows"] + c["valid_station_invalid_timestamp"]
                                + c["valid_station_outside_nominal_day"] + c["retained_rows"])


In [ ]:
required_artifacts = ["spatial/spatial_summary.json", "spatial/station_road_alignment_summary.json", "spatial/station_road_alignment.csv"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(11)
    print("미생성 단계: 셀 11 |", ", ".join(missing_artifacts))
else:
    spatial = read_json("spatial/spatial_summary.json")
    road = spatial["road_centerline"]
    dem = spatial["dem"]
    display(pd.DataFrame([
        {"노선": route, "중심선 길이 km": round(length, 3)}
        for route, length in road["route_length_km"].items()
    ]))
    print(f"중심선: {road['features']:,}개, CRS {road['crs']}, 무효 geometry {road['invalid_geometries']}개")
    print(f"DEM: {dem['width']:,}×{dem['height']:,} 픽셀, 해상도 {dem['pixel_scale'][0]:.0f}m, NoData {dem['nodata']}")
    print("DEM의 수직 고도 단위:", dem["vertical_unit_code"] or "메타데이터 미표기")

    alignment = read_json("spatial/station_road_alignment_summary.json")
    alignment_table = pd.read_csv(PRIVATE / "spatial" / "station_road_alignment.csv",
                                  dtype={"station": str})
    display(alignment_table[["station", "route", "distance_to_expected_m",
                             "nearest_matches_route"]].round(1))
    assert alignment["stations"] == 14
    assert alignment["nearest_route_mismatch"] == 0
    assert not (set(alignment_table.station) & PROTECTED)


## 2. 개발용 결빙 분포와 검증 분할

분모는 유효 상태 코드가 있는 개발용 RWIS 14곳의 10분 관측이다. 보호 3곳의 동적 값은 포함하지 않는다. 연속된 Ice 시점은 사건 단위로도 센다.

In [ ]:
required_artifacts = ["development/eda_summary.json", "development/monthly_eda.csv", "development/station_eda.csv"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(13)
    print("미생성 단계: 셀 13 |", ", ".join(missing_artifacts))
else:
    eda = read_json("development/eda_summary.json")
    monthly = pd.read_csv(PRIVATE / "development" / "monthly_eda.csv")
    stations = pd.read_csv(PRIVATE / "development" / "station_eda.csv", dtype={"station_id": str})
    print(f"유효 상태 {eda['valid_label']:,}건 중 Ice {eda['ice_slots']:,}건 "
          f"({eda['ice_slots']/eda['valid_label']*100:.4f}%)")
    print(f"Ice 사건 {eda['ice_events']}건, 단일 10분 사건 {eda['single_slot_events']}건")
    display(stations[["station_id", "route", "valid_label", "ice_slots", "ice_events"]]
            .sort_values("ice_slots", ascending=False).reset_index(drop=True))
    monthly["ice_per_10000"] = monthly["ice_slots"] / monthly["valid_label"] * 10000
    display(monthly[["month", "valid_label", "ice_slots", "ice_per_10000"]])
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.bar(monthly["month"], monthly["ice_per_10000"], color="#4477aa")
    ax.set(ylabel="Ice slots / 10,000 valid slots", title="Development RWIS monthly Ice rate")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    plt.show()
    assert not (set(stations["station_id"]) & PROTECTED)


In [ ]:
required_artifacts = ["validation/splits.json"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(14)
    print("미생성 단계: 셀 14 |", ", ".join(missing_artifacts))
else:
    splits = read_json("validation/splits.json")
    outer = splits["outer_folds"]
    for fold in outer:
        held = fold["validation_station"]
        train = set(fold["training_stations"])
        assert held not in train and not (train & PROTECTED)
        inner = [set(part["stations"]) for part in fold["inner_folds"]]
        assert set.union(*inner) == train
        assert all(not (a & b) for i, a in enumerate(inner) for b in inner[i + 1:])
    display(pd.DataFrame([
        {"보류 RWIS": fold["validation_station"],
         "학습 지점": len(fold["training_stations"]),
         "내부 분할 양성 지점 수": [part["positive_stations"] for part in fold["inner_folds"]]}
        for fold in outer
    ]))
    print("공간 외부 검증:", len(outer), "회 | 보호 지점 중첩: 0")


## 3. 과거 ASOS만 사용한 노면온도 기준 실험

각 목표 시각보다 **엄격히 이전**인 최근접 ASOS 기온(최대 60분 전)을 사용한다. 보정값은 해당 보류 RWIS를 제외한 13개 학습 지점의 중앙값이다. BIAS는 예측−관측이며 아래 표는 지점별 점수의 평균이다. 노면온도 단위와 ASOS 발표 지연은 확인이 필요하다.

In [ ]:
required_artifacts = ["baseline/summary.json", "baseline/station_metrics.csv"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(16)
    print("미생성 단계: 셀 16 |", ", ".join(missing_artifacts))
else:
    baseline = read_json("baseline/summary.json")
    metrics = pd.read_csv(PRIVATE / "baseline" / "station_metrics.csv",
                          dtype={"station": str, "nearest_asos": str})
    summary = pd.DataFrame([
        {"방법": "최근접 ASOS 그대로",
         "BIAS °C": baseline["direct_station_macro_bias"],
         "MAE °C": baseline["direct_station_macro_mae"],
         "RMSE °C": baseline["direct_station_macro_rmse"]},
        {"방법": "다른 지점의 중앙값 보정",
         "BIAS °C": baseline["adjusted_station_macro_bias"],
         "MAE °C": baseline["adjusted_station_macro_mae"],
         "RMSE °C": baseline["adjusted_station_macro_rmse"]},
    ])
    display(summary.round(3))
    display(metrics[["station", "nearest_asos", "distance_km", "direct_n",
                     "direct_mae", "adjusted_mae", "adjusted_rmse"]].round(3))
    print("유효 쌍:", f"{baseline['total_valid_pairs']:,}/{baseline['total_targets']:,}",
          "| 보정 후 MAE 악화 지점:", baseline["offset_worsened_stations"])
    fig, ax = plt.subplots(figsize=(7, 5))
    ordered = metrics.sort_values("adjusted_mae").reset_index(drop=True)
    y = np.arange(len(ordered))
    for i, row in ordered.iterrows():
        ax.plot([row["direct_mae"], row["adjusted_mae"]], [i, i], color="#bbbbbb")
    ax.scatter(ordered["direct_mae"], y, label="Direct ASOS", color="#4477aa", zorder=3)
    ax.scatter(ordered["adjusted_mae"], y, label="LOSO adjusted", color="#cc6677", zorder=3)
    ax.set(yticks=y, yticklabels=ordered["station"], xlabel="Road temperature MAE (°C)",
           ylabel="Held-out RWIS", title="Station-held-out baseline")
    ax.legend()
    fig.tight_layout()
    plt.show()


In [ ]:
required_artifacts = ["baseline/predictions.csv", "baseline/station_metrics.csv", "baseline/summary.json", "validation/splits.json"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(17)
    print("미생성 단계: 셀 17 |", ", ".join(missing_artifacts))
else:
    pred = pd.read_csv(PRIVATE / "baseline" / "predictions.csv",
                       usecols=["station", "time", "source_time", "age_minutes"],
                       dtype={"station": str})
    target_time = pd.to_datetime(pred["time"])
    source_time = pd.to_datetime(pred["source_time"])
    valid = source_time.notna()
    assert len(metrics) == len(splits["outer_folds"]) == 14
    assert not (set(pred["station"]) & PROTECTED)
    assert len(pred) == baseline["total_targets"]
    assert int(valid.sum()) == baseline["total_valid_pairs"]
    assert (source_time[valid] < target_time[valid]).all()
    assert pred.loc[valid, "age_minutes"].between(10, 60).all()
    assert np.isclose(metrics["direct_mae"].mean(), baseline["direct_station_macro_mae"])
    assert np.isclose(metrics["adjusted_mae"].mean(), baseline["adjusted_station_macro_mae"])
    print("보호 지점 제외, 과거 시각 입력, 10~60분 유효 범위, 지점 평균 점수 재계산: 모두 일치")


## 4. 과거 AWS·ASOS 기상 융합

원본 관측값을 화면에 나열하지 않고 연결률과 관측소별 OOF 오차만 표시한다. Ridge는 13개 학습 관측소에서만 결측 중앙값·스케일러를 맞추고 보류 관측소에 적용했다. 결과는 같은 ASOS 유효 시점에서 비교한다. 아래 −2~2°C 결과는 관측값으로 고른 **사후 진단 구간**이다.

In [ ]:
required_artifacts = ["baseline/station_metrics.csv", "baseline/summary.json", "weather_features/station_coverage.csv", "weather_features/summary.json", "weather_ridge/diagnostic_summary.json", "weather_ridge/station_metrics.csv", "weather_ridge/summary.json", "weather_ridge/winter_availability_diagnostic.csv"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(19)
    print("미생성 단계: 셀 19 |", ", ".join(missing_artifacts))
else:
    weather_features = read_json("weather_features/summary.json")
    weather_model = read_json("weather_ridge/summary.json")
    coverage = pd.read_csv(PRIVATE / "weather_features" / "station_coverage.csv", dtype={"station": str})
    weather_metrics = pd.read_csv(PRIVATE / "weather_ridge" / "station_metrics.csv", dtype={"station": str})
    print("기상 연결률: AWS", f"{weather_features['aws_temp_pairs']:,}/{weather_features['targets']:,}",
          "| ASOS", f"{weather_features['asos_temp_pairs']:,}/{weather_features['targets']:,}")
    display(pd.DataFrame([
        {"방법": "ASOS 보정", "지점 평균 MAE °C": weather_model["asos_adjusted_station_macro_mae"]},
        {"방법": "과거 기상 융합 Ridge", "지점 평균 MAE °C": weather_model["weather_ridge_station_macro_mae"]},
    ]).round(3))
    comparison = weather_metrics[["station", "n", "aws_available", "asos_adjusted_mae",
                                  "weather_ridge_mae", "near_freezing_n",
                                  "asos_adjusted_near_mae", "weather_ridge_near_mae"]].copy()
    comparison["MAE 차이 °C"] = comparison.weather_ridge_mae - comparison.asos_adjusted_mae
    display(comparison.round(3).sort_values("MAE 차이 °C"))
    fig, ax = plt.subplots(figsize=(8, 4))
    ordered = comparison.sort_values("MAE 차이 °C")
    ax.barh(ordered.station, ordered["MAE 차이 °C"],
            color=np.where(ordered["MAE 차이 °C"] > 0, "#cc6677", "#4477aa"))
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set(xlabel="Weather Ridge MAE − ASOS baseline MAE (°C)",
           title="Development station-held-out comparison")
    fig.tight_layout()
    plt.show()
    assert weather_features["targets"] == baseline["total_targets"]
    assert weather_features["asos_temp_pairs"] == baseline["total_valid_pairs"]
    assert weather_model["paired_rows"] == baseline["total_valid_pairs"]
    assert not (set(weather_metrics.station) & PROTECTED)
    assert np.isclose(weather_metrics.asos_adjusted_mae.mean(),
                      weather_model["asos_adjusted_station_macro_mae"])
    assert np.isclose(weather_metrics.weather_ridge_mae.mean(),
                      weather_model["weather_ridge_station_macro_mae"])
    print("기상 모델 MAE 악화 지점 수:", int((weather_metrics.weather_ridge_mae > weather_metrics.asos_adjusted_mae).sum()))

    diagnostic = read_json("weather_ridge/diagnostic_summary.json")
    diagnostic_table = pd.read_csv(PRIVATE / "weather_ridge" / "winter_availability_diagnostic.csv",
                                   dtype={"station": str})
    print("전체 MAE 악화 지점:", diagnostic["weather_worse_stations"],
          "| −2~2°C 진단 구간 악화 지점:", diagnostic["near_freezing_worse_stations"])
    display(diagnostic_table.loc[diagnostic_table.station.eq("12209"),
                                 ["winter_start_year", "aws_available", "n",
                                  "baseline_mae", "weather_mae",
                                  "near_freezing_n", "baseline_near_mae", "weather_near_mae"]].round(3))
    assert diagnostic["rows"] == weather_model["paired_rows"]


In [ ]:
required_artifacts = ["baseline/station_metrics.csv", "baseline/summary.json", "weather_features/features.parquet", "weather_features/station_coverage.csv", "weather_features/summary.json", "weather_ridge/diagnostic_summary.json", "weather_ridge/predictions.parquet", "weather_ridge/station_metrics.csv", "weather_ridge/summary.json", "weather_ridge/winter_availability_diagnostic.csv"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(20)
    print("미생성 단계: 셀 20 |", ", ".join(missing_artifacts))
else:
    feature_times = pd.read_parquet(PRIVATE / "weather_features" / "features.parquet",
                                    columns=["station", "time", "aws_source_time", "asos_source_time"])
    predicted = pd.read_parquet(PRIVATE / "weather_ridge" / "predictions.parquet",
                                columns=["station", "road_temp", "pred_weather_ridge"])
    assert not (set(feature_times.station.astype(str)) & PROTECTED)
    for name in ("aws_source_time", "asos_source_time"):
        present = feature_times[name].notna()
        assert (feature_times.loc[present, name] < feature_times.loc[present, "time"]).all()
    errors = (predicted.pred_weather_ridge - predicted.road_temp).abs()
    recomputed = errors.groupby(predicted.station.astype(str)).mean().sort_index()
    reported = weather_metrics.set_index("station").weather_ridge_mae.sort_index()
    assert np.allclose(recomputed.to_numpy(), reported.to_numpy())
    print("보호 지점 제외, 미래·동시각 관측값 제외, OOF MAE 재계산: 모두 일치")


### 겨울 간 시간 일반화 보조 검증

첫 겨울 학습 후 둘째 겨울에 적용한다. **같은 RWIS 14곳이 학습과 검증에 모두 있으므로 새 도로 위치의 공간 일반화 점수와 합치지 않는다.** ASOS 보정 기준과 기상 Ridge를 둘째 겨울의 같은 유효 시점에서 비교한다.

In [ ]:
required_artifacts = ["baseline/station_metrics.csv", "baseline/summary.json", "weather_features/station_coverage.csv", "weather_features/summary.json", "weather_ridge/diagnostic_summary.json", "weather_ridge/station_metrics.csv", "weather_ridge/summary.json", "weather_ridge/winter_availability_diagnostic.csv", "weather_temporal/station_metrics.csv", "weather_temporal/summary.json"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(22)
    print("미생성 단계: 셀 22 |", ", ".join(missing_artifacts))
else:
    temporal = read_json("weather_temporal/summary.json")
    temporal_stations = pd.read_csv(PRIVATE / "weather_temporal" / "station_metrics.csv",
                                    dtype={"station": str})
    display(pd.DataFrame([
        {"방법": "첫 겨울 ASOS 보정 → 둘째 겨울", "둘째 겨울 지점 평균 MAE °C": temporal["asos_adjusted_station_macro_mae"]},
        {"방법": "첫 겨울 기상 Ridge → 둘째 겨울", "둘째 겨울 지점 평균 MAE °C": temporal["weather_ridge_station_macro_mae"]},
    ]).round(3))
    display(temporal_stations[["station", "n", "asos_adjusted_mae", "weather_ridge_mae"]].round(3))
    assert temporal["status"] == "supplementary_temporal_transfer_only_not_spatial_holdout"
    assert temporal["train_rows"] + temporal["validation_rows"] == weather_model["paired_rows"]
    assert set(temporal_stations.station) == set(weather_metrics.station)
    assert not (set(temporal_stations.station) & PROTECTED)
    assert np.isclose(temporal_stations.weather_ridge_mae.mean(),
                      temporal["weather_ridge_station_macro_mae"])


## 해석과 다음 실행

ASOS 보정 기준 지점 평균 MAE 2.174°C에 비해 과거 AWS·ASOS 기상정보를 결합한 Ridge는 1.771°C였다. 이는 **개발용 14지점 공간 검증**이며 공식 3개소 점수나 조기예보 성능이 아니다. 첫 겨울 학습→둘째 겨울 보조 검증은 같은 지점에서 2.215→1.766°C였으며 공간 점수와 별도로 해석한다. 12209 지점과 −2~2°C 구간 일부 지점은 악화돼 후속 분석이 필요하다. 개발 RWIS 14곳은 제공 도로 중심선에서 모두 23m 이내였다. 다음에는 AWS 결측 대체 방식과 교량·터널·고도 같은 배포 가능한 정적 특징을 같은 분할에서 순차 비교한다.

세부 사항: [기상 결합 실험](../docs/research/WEATHER_FUSION_REPORT.md), [대회 요구사항](../docs/competition/COMPETITION_REQUIREMENTS.md), [선행자료](../docs/research/REFERENCES.md), [검증 규약](../docs/validation/VALIDATION_PROTOCOL.md), [운영진 질문](../docs/competition/QUESTIONS_FOR_ORGANIZER.md), [현재 상태](../docs/history/PROJECT_STATUS.md).

### AWS fallback availability: official-reference follow-up case

The development-only audit checks the three nearest AWS stations using valid readings strictly before each RWIS time and at most 60 minutes old. It reports input coverage, not model skill. Read [the evidence and next-test matrix](../docs/research/FOLLOWUP_EVIDENCE.md) and [primary references](../docs/research/REFERENCES.md). No held-out RWIS labels are used.

In [ ]:
required_artifacts = ["aws_fallback/summary.json", "aws_fallback/station_coverage.csv", "aws_fallback/nearest_aws.csv"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(25)
    print("미생성 단계: 셀 25 |", ", ".join(missing_artifacts))
else:
    fallback = read_json("aws_fallback/summary.json")
    assert fallback["status"] == "development_feature_availability_only"
    assert fallback["station_count"] == 14
    totals = fallback["totals"]
    coverage = pd.DataFrame([
        {"input": "existing nearest AWS", "available": totals["existing_nearest_aws_times"]},
        {"input": "top 3 AWS with prior valid readings", "available": totals["top3_any_valid_times"]},
    ])
    coverage["target_times"] = totals["target_times"]
    coverage["coverage_percent"] = 100 * coverage["available"] / coverage["target_times"]
    display(coverage)
    display(pd.DataFrame([fallback["station_12209"]]))
    print("Other AWS rescue:", totals["other_aws_rescue_times"],
          "| still missing:", totals["still_missing_times"])

    station_coverage = pd.read_csv(PRIVATE / "aws_fallback" / "station_coverage.csv",
                                   dtype={"rwis_station": str})
    assert not set(station_coverage["rwis_station"]) & PROTECTED
    station_coverage["existing_percent"] = (100 * station_coverage["existing_nearest_aws_times"]
                                            / station_coverage["target_times"]).round(1)
    station_coverage["top3_percent"] = (100 * station_coverage["top3_any_valid_times"]
                                        / station_coverage["target_times"]).round(1)
    display(station_coverage[["rwis_station", "existing_percent", "top3_percent",
                              "other_aws_rescue_times", "still_missing_times"]]
            .sort_values("other_aws_rescue_times", ascending=False))
    aws_candidates = pd.read_csv(PRIVATE / "aws_fallback" / "nearest_aws.csv",
                                 dtype={"rwis_station": str, "aws_station": str})
    display(aws_candidates.loc[aws_candidates.rwis_station.eq("12209"),
                               ["aws_rank", "aws_station", "distance_km",
                                "valid_prior_times", "median_age_minutes_when_valid"]])


### Paired LOSO result: nearest versus fallback AWS

The same target rows, held-out stations, Ridge settings, and ASOS baseline are used for both variants. This is development evidence only. The [case matrix](../docs/research/FOLLOWUP_EVIDENCE.md) records failures and literature limits.

In [ ]:
required_artifacts = ["baseline/station_metrics.csv", "baseline/summary.json", "weather_fallback_comparison/station_metrics.csv", "weather_fallback_comparison/summary.json", "weather_features/station_coverage.csv", "weather_features/summary.json", "weather_ridge/diagnostic_summary.json", "weather_ridge/station_metrics.csv", "weather_ridge/summary.json", "weather_ridge/winter_availability_diagnostic.csv"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(27)
    print("미생성 단계: 셀 27 |", ", ".join(missing_artifacts))
else:
    variant = read_json("weather_fallback_comparison/summary.json")
    assert variant["paired_rows"] == weather_model["paired_rows"]
    display(pd.DataFrame([{"model": "nearest AWS", "macro_MAE_C": variant["nearest_macro_mae"],
                           "macro_RMSE_C": variant["nearest_macro_rmse"]},
                          {"model": "fallback AWS", "macro_MAE_C": variant["fallback_macro_mae"],
                           "macro_RMSE_C": variant["fallback_macro_rmse"]}]))
    variant_station = pd.read_csv(PRIVATE / "weather_fallback_comparison" / "station_metrics.csv",
                                  dtype={"station": str})
    assert not set(variant_station.station) & PROTECTED
    display(variant_station[["station", "n", "nearest_mae", "fallback_mae",
                             "near_freezing_n", "nearest_near_mae", "fallback_near_mae"]]
            .sort_values("station"))
    print("Worse near freezing:", variant["stations_fallback_near_mae_worse"])
    display(pd.DataFrame([variant["station_12209"]]))


## Competition evaluation scorecard and next runs

Read the Korean [evaluation status and execution order](../docs/validation/EVALUATION_STATUS.md). The protected RWIS stations 12410, 33411, and 23402 remain reserved for one final test. Current numeric results cover development-site road-temperature estimation and Ice classification only. A development-only 10-minute Ice baseline and provisional daily scores now exist, but neither input variant is operationally adequate. No official daily score or three-route risk map has been produced. Next, examine one-slot Ice labels, prior moisture history, station-balanced threshold objectives, and spatial features using the same held-out-station folds.

## Development Ice classification: first complete baseline

This is a provisional evaluation on the 14 development RWIS stations. The three official held-out stations remain protected. The model uses only prior weather inputs; the stored score is not a calibrated probability. Read [the failure analysis and run commands](../docs/research/ICE_BASELINE_REPORT.md).

In [ ]:
required_artifacts = ["ice_features/summary.json", "ice_logistic/summary.json", "ice_fallback_logistic/summary.json", "ice_variant_comparison/summary.json", "ice_diagnostic/summary.json"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(30)
    print("미생성 단계: 셀 30 |", ", ".join(missing_artifacts))
else:
    ice_inputs = read_json("ice_features/summary.json")
    ice_nearest = read_json("ice_logistic/summary.json")
    ice_fallback = read_json("ice_fallback_logistic/summary.json")
    ice_comparison = read_json("ice_variant_comparison/summary.json")
    ice_diagnostic = read_json("ice_diagnostic/summary.json")
    assert ice_inputs["valid_labels"] == ice_comparison["paired_valid_labels"]
    assert ice_inputs["ice_slots"] == ice_comparison["ice_slots"]
    rows = []
    for name, model in [("nearest AWS", ice_nearest), ("fallback AWS", ice_fallback)]:
        micro = model["slot_micro"]
        rows.append({"input": name, "TP": micro["tp"], "FP": micro["fp"],
                     "FN": micro["fn"], "precision": micro["precision"],
                     "recall": micro["recall"], "F1": micro["f1"],
                     "defined_daily_F1_days": model["daily_10minute_station_macro"]["defined_f1_days"]})
    display(pd.DataFrame(rows))
    display(pd.DataFrame(ice_comparison["fixed_observed_ice_days"]).T)
    print("Accepted complete station-days:", ice_comparison["accepted_station_days"])
    print("Ice AWS air-temperature coverage:",
          ice_diagnostic["ice_nearest_aws_temp_available"], "to",
          ice_diagnostic["ice_fallback_aws_temp_available"])


In [ ]:
required_artifacts = ["ice_variant_comparison/station_metrics.csv"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(31)
    print("미생성 단계: 셀 31 |", ", ".join(missing_artifacts))
else:
    ice_station = pd.read_csv(PRIVATE / "ice_variant_comparison" / "station_metrics.csv",
                              dtype={"station": str})
    assert not set(ice_station.station) & PROTECTED
    display(ice_station[["station", "ice_slots", "nearest_tp", "nearest_fp",
                         "fallback_tp", "fallback_fp"]].sort_values("station"))
    print("Uncalibrated classifier scores and provisional daily policy; do not use as final score.")


### One-slot Ice event context audit

The development-only audit examines the immediately previous and next 10-minute road-state codes for each Ice run. It does not relabel or discard Ice. The event-level file is private; only aggregate counts are shown here.

In [ ]:
required_artifacts = ["ice_event_audit/summary.json", "development/eda_summary.json"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(33)
    print("미생성 단계: 셀 33 |", ", ".join(missing_artifacts))
else:
    ice_events = read_json("ice_event_audit/summary.json")
    assert ice_events["events"] == read_json("development/eda_summary.json")["ice_events"]
    assert ice_events["single_slot_events"] == read_json("development/eda_summary.json")["single_slot_events"]
    display(pd.DataFrame([ice_events]).T.rename(columns={0: "count_or_note"}))


## Facility proximity and road-temperature estimation

Two exploratory definitions are compared: distance to any supplied bridge/tunnel polygon, and distance only to polygons intersecting a 50 m buffer around the station's highway route. Both use the same held-out stations and timestamps. The source vintage during the observation winters is unverified.

In [ ]:
required_artifacts = ["spatial/station_facility/summary.json", "spatial/station_facility_route/summary.json", "facility_weather_comparison/summary.json", "route_facility_weather_comparison/summary.json", "route_facility_weather_comparison/station_metrics.csv", "aws_fallback_ridge/summary.json"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(35)
    print("미생성 단계: 셀 35 |", ", ".join(missing_artifacts))
else:
    facility = read_json("spatial/station_facility/summary.json")
    route_facility = read_json("spatial/station_facility_route/summary.json")
    ablation = read_json("facility_weather_comparison/summary.json")
    route_ablation = read_json("route_facility_weather_comparison/summary.json")
    assert facility["station_count"] == route_facility["station_count"] == 14
    assert ablation["paired_rows"] == route_ablation["paired_rows"] == read_json("aws_fallback_ridge/summary.json")["paired_rows"]
    facility_station = pd.read_csv(PRIVATE / "route_facility_weather_comparison" / "station_metrics.csv", dtype={"station": str})
    assert not set(facility_station.station) & PROTECTED
    display(pd.DataFrame([{"variant": "weather only", "macro_MAE_C": ablation["weather_macro_mae"], "stations_improved": None}, {"variant": "any facility", "macro_MAE_C": ablation["facility_macro_mae"], "stations_improved": ablation["stations_facility_mae_better"]}, {"variant": "route-aligned facility", "macro_MAE_C": route_ablation["facility_macro_mae"], "stations_improved": route_ablation["stations_facility_mae_better"]}]))
    display(facility_station[["station", "bridge_distance_m", "tunnel_distance_m", "weather_mae", "facility_mae", "weather_near_mae", "facility_near_mae"]].round(3))

## Prior three-hour AWS context and Ice

The 15-minute precipitation windows overlap, so their values are not added into a three-hour rainfall total. The table is descriptive, not causal; station, season and temporal dependence can confound the apparent association.

In [ ]:
required_artifacts = ["ice_weather_history/summary.json", "development/eda_summary.json"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(37)
    print("미생성 단계: 셀 37 |", ", ".join(missing_artifacts))
else:
    history = read_json("ice_weather_history/summary.json")
    e = history["exposure"]
    assert history["ice_slots"] == read_json("development/eda_summary.json")["ice_slots"]
    assert history["positive_history_available"] <= history["ice_slots"]
    rate = pd.DataFrame([{"condition": "AWS air -5..3 C; any wet observation in prior 3h", "slots": e["cool_wet3h_rows"], "ice_slots": e["cool_wet3h_ice_slots"]}, {"condition": "AWS air -5..3 C; no wet observation in prior 3h", "slots": e["cool_dry3h_rows"], "ice_slots": e["cool_dry3h_ice_slots"]}])
    rate["ice_fraction"] = rate["ice_slots"] / rate["slots"]
    display(rate)
    print("Positive Ice slots with prior AWS history:", history["positive_history_available"], "of", history["ice_slots"])

### Development LOSO classification with prior weather history

For each outer held-out station, the threshold is selected only from an inner station-level OOF prediction on the other stations. The class-weighted logistic score is not a calibrated probability. Daily metrics remain provisional pending organizer guidance.

In [ ]:
required_artifacts = ["ice_history_logistic/summary.json", "ice_fallback_logistic/summary.json", "ice_history_comparison/summary.json", "ice_history_comparison/station_metrics.csv", "development/eda_summary.json"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(39)
    print("미생성 단계: 셀 39 |", ", ".join(missing_artifacts))
else:
    hist_model = read_json("ice_history_logistic/summary.json")
    base_model = read_json("ice_fallback_logistic/summary.json")
    paired_history = read_json("ice_history_comparison/summary.json")
    assert hist_model["target_slots"] == base_model["target_slots"] == paired_history["paired_target_slots"]
    assert hist_model["valid_label_slots"] == base_model["valid_label_slots"] == paired_history["paired_valid_labels"]
    assert hist_model["ice_slots"] == base_model["ice_slots"] == read_json("development/eda_summary.json")["ice_slots"]
    rows = []
    for name, key in [("Current AWS", "fallback"), ("Plus 3h history", "history")]:
        m = paired_history[key + "_slot_micro"]
        fixed = paired_history["fixed_observed_ice_days"][key]
        rows.append({"model": name, "TP": m["tp"], "FP": m["fp"], "FN": m["fn"], "precision": m["precision"], "recall": m["recall"], "F1": m["f1"], "fixed_observed_Ice_days_F1": fixed["mean_f1_equal_days"], "false_alarm_noIce_days": fixed["false_alarm_nonice_station_days"]})
    display(pd.DataFrame(rows))
    station_history = pd.read_csv(PRIVATE / "ice_history_comparison" / "station_metrics.csv", dtype={"station": str})
    assert not set(station_history.station) & PROTECTED
    display(station_history[["station", "ice_slots", "fallback_tp", "fallback_fp", "history_tp", "history_fp"]])

## Prior same-route RWIS benchmark

For each development target, take the nearest **other** development RWIS on the same highway route. Only road temperature measured strictly before the target, within 30 minutes, is usable. The three official protected RWIS are excluded from every source set. Compare the nonparametric neighbor value, the existing weather-only outer-LOSO Ridge, and their fixed 1:1 average on identical rows. This remains a development benchmark; source data latency and coverage on uninstrumented segments require separate checks.

In [ ]:
required_artifacts = ["neighbor_rwis/summary.json", "neighbor_rwis/diagnostic_summary.json", "neighbor_rwis/station_metrics.csv"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(41)
    print("미생성 단계: 셀 41 |", ", ".join(missing_artifacts))
else:
    neighbor = read_json("neighbor_rwis/summary.json")
    neighbor_diag = read_json("neighbor_rwis/diagnostic_summary.json")
    neighbor_station = pd.read_csv(PRIVATE / "neighbor_rwis" / "station_metrics.csv", dtype={"station": str, "source_station": str})
    assert neighbor["target_stations"] == 14
    assert neighbor["paired_rows"] == neighbor_diag["paired_rows"]
    assert not set(neighbor_station.station) & PROTECTED
    assert not set(neighbor_station.source_station) & PROTECTED
    assert not (neighbor_station.station == neighbor_station.source_station).any()
    display(pd.DataFrame([{"input": "Prior AWS/ASOS Ridge", "macro_MAE_C": neighbor["weather_macro_mae_on_paired"], "macro_RMSE_C": neighbor["weather_macro_rmse_on_paired"], "near_freezing_MAE_C": neighbor["weather_macro_near_mae_on_paired"]}, {"input": "Prior nearest RWIS", "macro_MAE_C": neighbor["neighbor_macro_mae"], "macro_RMSE_C": neighbor["neighbor_macro_rmse"], "near_freezing_MAE_C": neighbor["neighbor_macro_near_mae"]}, {"input": "Fixed equal average", "macro_MAE_C": neighbor["equal_blend_macro_mae"], "macro_RMSE_C": neighbor["equal_blend_macro_rmse"], "near_freezing_MAE_C": neighbor["equal_blend_macro_near_mae"]}]))
    display(neighbor_station[["station", "source_station", "source_distance_km", "paired_rows", "weather_mae", "neighbor_mae", "blend_mae"]].round(3))
    print("Source observation age counts (minutes):", neighbor_diag["age_minutes_counts"])

In [ ]:
required_artifacts = ["neighbor_rwis/summary.json", "neighbor_rwis/diagnostic_summary.json", "neighbor_rwis/station_metrics.csv"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(42)
    print("미생성 단계: 셀 42 |", ", ".join(missing_artifacts))
else:
    winter_rows = []
    for name, values in neighbor_diag["winters"].items():
        winter_rows.append({"winter": name, **values})
    display(pd.DataFrame(winter_rows))
    route_rows = []
    for name, values in neighbor_diag["routes"].items():
        route_rows.append({"route": name, **values})
    display(pd.DataFrame(route_rows))
    print("Both winter summaries stratify the same spatial-LOSO weather fits; they are not winter-forward validation.")

## 2026-10-02: 다음 겨울·미관측 지점 및 RWIS 지연 검증

구버전 20260915 개발 검증입니다. 공식 수정본 배포 후 재실행해야 합니다. 고정 1:1 혼합이며 보호 3개 지점은 사용하지 않습니다. OFFICIAL_UPDATE_20261002.md와 NEIGHBOR_TRANSFER_REPORT.md에서 공식 변경사항과 상세 설계를 확인하세요.


In [ ]:
required_artifacts = ["neighbor_transfer/summary.json", "neighbor_transfer/station_metrics.csv", "neighbor_transfer/fold_manifest.json"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(44)
    print("미생성 단계: 셀 44 |", ", ".join(missing_artifacts))
else:
    transfer = read_json("neighbor_transfer/summary.json")
    transfer_metrics = pd.read_csv(PRIVATE / "neighbor_transfer/station_metrics.csv", dtype={"station": str, "source_station": str})
    folds = read_json("neighbor_transfer/fold_manifest.json")
    assert len(folds) == 14
    assert not set(transfer_metrics.station) & PROTECTED
    assert not set(transfer_metrics.source_station) & PROTECTED
    for fold in folds:
        assert fold["held_station"] not in fold["train_stations"]
        assert fold["held_station"] != fold["reference_station"]
        assert not set(fold["train_stations"]) & PROTECTED
        assert pd.Timestamp(fold["train_end"]) < pd.Timestamp(fold["valid_start"])
    assert len({s["rows"] for s in transfer["scenarios"]}) == 1
    display(pd.DataFrame(transfer["scenarios"]))
    display(transfer_metrics[["station", "delay_minutes", "weather_mae", "blend_mae", "neighbor_available_rows"]])
    print(transfer["limitations"])


## 비선형 결빙 분류 및 임계값 검증

동일 기상 입력·LOSO 분할의 개발 실험입니다. 내부 OOF에서만 임계값을 정하고 외부 지점으로 비교합니다. 현재 작업 공간에 고정한 원본 버전의 결과이며 공식 최종 점수가 아닙니다. `ICE_NONLINEAR_REPORT.md`에 오경보·미탐과 채택 여부를 기록합니다. 모델 재학습은 시간이 걸리므로 기본 노트북은 저장된 결과만 확인합니다.

In [ ]:
required_artifacts = ["ice_nonlinear/summary.json", "ice_nonlinear_comparison/summary.json", "ice_nonlinear_comparison/station_metrics.csv", "ice_nonlinear/fold_manifest.json", "ice_features/summary.json", "development/eda_summary.json"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(46)
    print("미생성 단계: 셀 46 |", ", ".join(missing_artifacts))
else:
    nonlinear = read_json("ice_nonlinear/summary.json")
    comparison = read_json("ice_nonlinear_comparison/summary.json")
    assert nonlinear["target_rows"] == comparison["target_rows"] == read_json("ice_features/summary.json")["target_slots"]
    assert comparison["valid_label_rows"] == read_json("ice_features/summary.json")["valid_labels"]
    assert comparison["ice_slots"] == read_json("development/eda_summary.json")["ice_slots"]
    station_table = pd.read_csv(PRIVATE / "ice_nonlinear_comparison/station_metrics.csv", dtype={"station": str})
    assert not set(station_table.station) & PROTECTED
    for fold in read_json("ice_nonlinear/fold_manifest.json"):
        fit = set(fold["fit_stations"])
        check = set(fold["validation_stations"])
        assert not fit & check
        assert fold["outer_held"] not in fit | check
        assert not (fit | check) & PROTECTED
    rows = []
    for name, result in comparison["variants"].items():
        assert result["exact_episode_diagnostic"]["observed_events"] == read_json("development/eda_summary.json")["ice_events"]
        rows.append({"variant": name, **result["slot_micro"],
                     "daily_station_macro_f1": result["daily_station_macro"]["f1"],
                     "fixed_positive_day_f1": result["fixed_ice_day_f1_station_first"],
                     "false_alarm_days": result["false_alarm_nonice_days"],
                     "detected_stations": result["stations_with_true_positives"]})
    display(pd.DataFrame(rows))
    display(station_table[["variant", "station", "positive_slots", "tp", "fp", "fn", "fixed_ice_day_f1"]])
    print(comparison["fixed_positive_day_station_bootstrap"])
    print(comparison["limitations"])


## 수정본 전 작업: 주변 관측과 도로 구간

PRE_CORRECTION_WORK_REPORT.md에 정보량·참조 결손·오경보·정적 구간 결과를 정리했습니다. outer_context는 감사용이며 모든 학습 분할에 공통 결합하면 안 됩니다. 구간 지도에는 위험 점수를 부여하지 않았습니다.

In [ ]:
required_artifacts = ["rwis_context/summary.json", "rwis_context/inner_reference_audit.csv", "rwis_context/station_coverage.csv", "rwis_context/error_context_summary.json", "ice_features/summary.json", "development/eda_summary.json"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(48)
    print("미생성 단계: 셀 48 |", ", ".join(missing_artifacts))
else:
    context = read_json("rwis_context/summary.json")
    assert context["target_rows"] == read_json("ice_features/summary.json")["target_slots"] and context["ice_slots"] == read_json("development/eda_summary.json")["ice_slots"]
    refs = pd.read_csv(PRIVATE / "rwis_context/inner_reference_audit.csv", dtype={"outer_held": str, "target": str, "source_station": str})
    assert len(refs) == 1092
    for row in refs.itertuples():
        if pd.notna(row.source_station):
            assert row.source_station not in set(row.excluded_stations.split(",")) | {row.target} | PROTECTED
    coverage = pd.read_csv(PRIVATE / "rwis_context/station_coverage.csv", dtype={"station": str, "source_station": str, "moisture_station": str})
    assert not set(coverage.station) & PROTECTED
    assert not set(coverage.source_station) & PROTECTED
    assert not set(coverage.moisture_station.dropna()) & PROTECTED
    display(coverage)
    print(context["fixed_latest_ice_benchmark"])
    print(context["fixed_past60_ice_benchmark"])
    print("Missing inner references:", context["inner_missing_reference_cases"])
    display(pd.DataFrame(read_json("rwis_context/error_context_summary.json")["condition_groups"]))


In [ ]:
required_artifacts = ["spatial/segment_scaffold/summary.json", "spatial/segment_scaffold/segment_inventory.csv"]
missing_artifacts = [name for name in required_artifacts if not (PRIVATE / name).exists()]
if missing_artifacts:
    SKIPPED_REVIEW_CELLS.append(49)
    print("미생성 단계: 셀 49 |", ", ".join(missing_artifacts))
else:
    scaffold = read_json("spatial/segment_scaffold/summary.json")
    assert scaffold["segments"] == 467
    assert abs(scaffold["source_length_m"] - scaffold["segment_length_m"]) < .001
    assert scaffold["roundtrip_gpkg_and_shapefile_verified"]
    inventory = pd.read_csv(PRIVATE / "spatial/segment_scaffold/segment_inventory.csv")
    assert inventory.seg_id.is_unique
    assert inventory.risk_scr.isna().all()
    assert inventory.risk_stat.eq("unscored").all()
    assert inventory.length_m.between(0,1000.000001,inclusive="right").all()
    display(pd.DataFrame(scaffold["routes"]))
    print(scaffold["limitations"])


## 현재 배포본의 결빙 기준 모델과 미생성 단계
전체 비교 실험을 아직 만들지 않았어도 완료된 기준 모델은 아래에서 확인한다. 미생성 단계가 있으면 전체 연구 재현 완료로 해석하지 않는다.

In [ ]:
model_path = PRIVATE / "ice_fallback_logistic/summary.json"
if model_path.exists():
    current_model = json.loads(model_path.read_text(encoding="utf-8"))
    assert current_model["stations"] == 14
    assert current_model["slot_micro"]["tp"] + current_model["slot_micro"]["fn"] == current_model["ice_slots"]
    display(pd.DataFrame([current_model["slot_micro"]]))
    display(pd.DataFrame([current_model["daily_10minute_station_macro"]]))
    print("개발용 결과이며 공식 보호 지점 점수가 아니다.")
else:
    print("결빙 기준 모델 미생성")
print("미생성 산출물로 생략한 검토 셀:", SKIPPED_REVIEW_CELLS)


## 수정본 센서 감사와 시정 입력 추가 비교
시정은 미터 단위 관측이며 습도로 변환하지 않는다. 같은 노선의 과거 30분 이내 자료만 연결하고 호남선은 결측으로 유지한다. 아래는 개발용 비교이며 공식 최종 평가가 아니다. [통합 검토](../docs/competition/OFFICIAL_TABS_REVIEW_20261007.md)를 함께 읽는다.

In [ ]:
for name in ["sensor_availability/summary.json", "ice_fog_features_delay0/summary.json", "ice_fog_comparison/summary.json"]:
    path = PRIVATE / name
    if path.exists():
        result = json.loads(path.read_text(encoding="utf-8"))
        print(name, result.get("status"))
        if "slot_metrics" in result:
            display(pd.DataFrame(result["slot_metrics"]).T)
            print("동일 타깃 행:", result["target_rows"])
        elif "available_rows" in result:
            print("전체/시정 가용/결빙 시정 가용:", result["rows"], result["available_rows"], result["ice_with_fog"])
        else:
            print(result.get("stations"))
    else:
        print("미생성 단계:", name)

## 문헌 기반 물리·지형 고도화 — 2026-10-08

열·수분 후보의 중첩 공간 검증과 다음 겨울 제외 지점 검증을 구별한다. NASA 복사량은 공개 지연 때문에 사후 추정 전용이며, 지형·일조는 DSM 기반 근삿값이다. 보호 3개 지점은 사용하지 않는다. [문헌 및 해석](../docs/research/FOLLOWUP_EVIDENCE.md)을 함께 읽는다.

In [ ]:
for name in ["physical_ice_comparison/summary.json", "solar_terrain_features/summary.json", "forward_physics/summary.json", "temperature_physics/summary.json", "daily_rank_policy/summary.json", "catboost_forward/summary.json"]:
    path = PRIVATE / name
    if not path.exists():
        print("미생성 단계:", name)
        continue
    result = json.loads(path.read_text(encoding="utf-8"))
    print(name, result.get("status"), "행 수:", result.get("rows"))
    variants = result.get("variants", result.get("metrics", {}))
    if variants:
        display(pd.DataFrame([{ "후보": key, **value["slot_micro"], "잠정 일평균 F1": value["daily_station_macro"]["f1"], "무결빙일 오경보": value["false_alarm_nonice_days"]} for key, value in variants.items()]))
    elif "station_macro" in result:
        display(pd.DataFrame.from_dict(result["station_macro"], orient="index"))
    elif "nonmissing" in result:
        display(pd.DataFrame.from_dict(result["nonmissing"], orient="index", columns=["가용 행"]))
    print("공식 최종 점수 또는 미래 예보 성능이 아닙니다.")


## 새 구간 추론과 수분 이력 검증

노선 이름 대응·과거 참조 입력을 전수 검증한 뒤 구간 추정값을 생성한다. 조건 노출은 실제 Ice 발생확률이나 공식 위험등급이 아니다. DSM 높이 대체로 발생한 성능 변화를 별도로 확인한다. 무효화된 최초 구간 출력은 사용하지 않는다.

In [ ]:
for name in ["memory_forward/summary.json", "height_transfer/summary.json", "dsm_consistent/summary.json", "point_sources/parity_verification.json", "segment_inference_v1/summary.json", "segment_inference_v1/audit.json", "segment_condition_proposal/manifest.json"]:
    path = PRIVATE / name
    if not path.exists():
        print("미생성 단계:", name)
        continue
    result = json.loads(path.read_text(encoding="utf-8"))
    print(name, result.get("status"), "행 수:", result.get("rows"), "구간:", result.get("segments"))
    if "metrics" in result:
        display(pd.DataFrame([{"후보": key, **value["slot_micro"], "잠정 일평균 F1": value["daily_station_macro"]["f1"], "고정 결빙일 F1": value["fixed_ice_day_f1"]} for key, value in result["metrics"].items()]))
    elif "feature_count" in result:
        print("입력 동등성 지점/특성:", result["stations"], result["feature_count"])
    elif "reference_stations" in result:
        print("참조 지점:", result["reference_stations"], "과거 참조 최대 거리(km):", result["max_reference_distance_km"])
print("후보의 고정 보호 평가 결과는 다음 절에서 확인합니다. 구간의 관측 정답이 없어 조건 노출 지도를 결빙 정확도로 해석하지 않습니다.")

## 고정 후 보호 지점 평가와 데모

보호 지점 평가 전에 예측을 저장하고 해시를 남겼다. 아래는 고정된 결과를 읽기만 한다. 보호 지점 평가를 다시 실행하거나 결과를 보고 임계값을 조정하지 않는다. 구간 분류는 DSM 대체의 탐색적 출력이며 위험등급에 사용하지 않는다.

In [ ]:
path = PRIVATE / "frozen_assessment_v1/summary.json"
if path.exists():
    final = json.loads(path.read_text(encoding="utf-8"))
    print(final["status"])
    display(pd.DataFrame([{"지점": s["station"], **s["temperature"], "Ice TP": s["ice_slot"]["tp"], "Ice FP": s["ice_slot"]["fp"], "Ice FN": s["ice_slot"]["fn"]} for s in final["stations"]]))
    print("온도 지점 평균:", final["temperature_station_macro"])
    print("결빙 지표:", final["ice_slot_micro"])
    print("운영진이 발급한 공식 점수가 아닙니다. 결빙 분류 실패를 온도 결과로 대체하지 않습니다.")
else:
    print("이 작업공간에는 고정 보호 지점 평가가 없습니다.")
for name in ["segment_ice_exploratory/summary.json", "demo/video_manifest.json"]:
    path = PRIVATE / name
    if path.exists():
        result = json.loads(path.read_text(encoding="utf-8"))
        print(name, result["status"])


## 도로 이정 교차 검토와 완료 감사

연구 참조의 기하 일치와 공식 CCTV 기준점 일치를 구분한다. 기준점 일치만으로 구간 전체 이정·방향을 확정하지 않는다. 참조 자료는 재배포 조건 미확인으로 비공개이며 제출 초안에서 제외한다. 완료 감사의 미완료 항목을 그대로 표시한다.

In [ ]:
for name in ["spatial/chainage_reference/summary.json", "spatial/official_chainage_anchors/summary.json", "completion_audit_v1/summary.json"]:
    path = PRIVATE / name
    if not path.exists():
        print("이 작업공간에 없는 비공개 점검:", name)
        continue
    result = json.loads(path.read_text(encoding="utf-8"))
    print(name, result["status"])
    if "requirements" in result:
        display(pd.DataFrame(result["requirements"])[["id", "status"]])
    elif "within_provided_segment_candidate_span" in result:
        display(pd.DataFrame(result["within_provided_segment_candidate_span"]))
    else:
        print("기하 통과 / 수동 검토:", result["passes_geometry_checks"], result["requires_manual_review"])
print("공식 전체 구간 정합성 및 검증된 결빙 위험등급은 아직 확보하지 못했습니다.")

## 논문 기반 단기 냉각 누적의 개발 비교

Yi·Seo(2026), arXiv:2609.33528의 개념을 10분 관측·다른 지점의 과거 참조에 맞춰 검증한다. 공식 04 양성 정의를 유지한다. 논문의 04/15 양성·1분 발생 예보와 다르므로 논문 성능을 재현했다고 표현하지 않는다. 반복 연구한 개발 자료의 비교이며 보호 지점을 다시 평가하지 않는다.

In [ ]:
path = PRIVATE / "antecedent_forward/summary.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    print(result["status"])
    display(pd.DataFrame([{"후보": key, **value["slot_micro"], "Average precision": value["average_precision"], "ROC AUC": value["roc_auc"], "잠정 일평균 F1": value["daily_station_macro"]["f1"], "실제 결빙일 F1": value["fixed_ice_day_f1"]} for key, value in result["metrics"].items()]))
else:
    print("이 작업공간에는 단기 누적 개발 비교가 아직 없습니다.")
print("고정 보호 평가의 F1=0 결과를 대체하는 독립 최종 성능이 아닙니다.")

## 규제 선형·가법 스플라인 개발 비교

고도를 제외한 같은 열·수분 입력에서 규제 로지스틱과4변수 가법 스플라인을 비교한다. 매듭 위치는 정답을 보기 전에 물리 단위로 고정했고 전처리는 각 학습 분할 안에서만 학습한다. 기존 HGB는 고도 입력을 포함하므로 모든 차이를 분류기 구조의 효과로 해석하지 않는다. 보호 지점은 재평가하지 않는다.

In [ ]:
path = PRIVATE / "regularized_forward/summary.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    print(result["status"])
    display(pd.DataFrame([{"후보": key, **value["slot_micro"], "Average precision": value["average_precision"], "잠정 일평균 F1": value["daily_station_macro"]["f1"], "실제 결빙일 F1": value["fixed_ice_day_f1"]} for key, value in result["metrics"].items()]))
else:
    print("이 작업공간에는 규제 모델 비교가 아직 없습니다.")
print("반복 연구한 개발 자료의 결과입니다. 기존 고정 보호 평가를 대체하지 않습니다.")

## 다른 지점의 과거 노면상태 추가 비교

공식04 양성 정의를 유지하고 같은 노선의 다른 개발 지점 과거 상태를 추가한다. 자기·내부/외부 제외·보호 지점의 상태와 동시각/미래 관측은 입력에서 차단한다.13·15는 별도 상태 입력이고 양성 정답으로 바꾸지 않는다.60/180분 관측 커버리지80% 미만의 비율은 미정의다. 새로운 독립 최종 검증이 아니다.

In [ ]:
path = PRIVATE / "surface_state_forward/summary.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    print(result["status"])
    display(pd.DataFrame([{"후보": key, **value["slot_micro"], "Average precision": value["average_precision"], "잠정 일평균 F1": value["daily_station_macro"]["f1"], "실제 결빙일 F1": value["fixed_ice_day_f1"]} for key, value in result["metrics"].items()]))
else:
    print("이 작업공간에는 과거 주변 노면상태 비교가 아직 없습니다.")
audit_path = PRIVATE / "surface_state_forward/audit.json"
if audit_path.exists():
    print(json.loads(audit_path.read_text(encoding="utf-8")))
print("공식 보호3지점의 고정 평가·모델·임계값은 변경하지 않습니다.")

## 고정 평균 결합의 개발 비교

산술·기하 평균의 가중치는 각각 0.5로 고정한다. 동일 대상의 첫 겨울 내부 OOF를 대조하고 그 자료에서만 임계값을 선택한다. 검증 겨울 결과로 가중치를 튜닝하지 않는다. 원점수와 평균은 보정된 결빙확률이 아니며 반복 개발 비교를 독립 최종 성능으로 해석하지 않는다.

In [ ]:
path = PRIVATE / "surface_state_ensemble/summary.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    print(result["status"])
    display(pd.DataFrame([{"후보": key, **value["slot_micro"], "Average precision": value["average_precision"], "잠정 일평균 F1": value["daily_station_macro"]["f1"], "실제 결빙일 F1": value["fixed_ice_day_f1"]} for key, value in result["metrics"].items()]))
else:
    print("이 작업공간에는 고정 평균 비교가 아직 없습니다.")
audit_path = PRIVATE / "surface_state_ensemble/audit.json"
if audit_path.exists():
    print(json.loads(audit_path.read_text(encoding="utf-8")))
print("오경보·미탐의 동시 개선을 달성하지 못해 채택하지 않았습니다.")

## 추정 노면온도와 수분차의 교차 적합 비교

Ridge α=100의 노면온도 추정을 분류 입력으로 사용한다. 분류 학습 행도 자기 지점 그룹을 제외해 온도를 예측하고, 내부 평가 지점은 해당 단계의 모든 온도 학습에서 제외한다. 외부 제외·보호 지점도 참조하지 않는다. 같은 분류 참조의 이전30분 이내 온도와0.5 혼합하며 ASOS가 없으면 미정의다. 온도 정답은 첫 단계 회귀 학습 및 개발 온도 진단에만 쓰고 분류 입력에서 제외한다.

In [ ]:
path = PRIVATE / "estimated_temperature_forward/summary.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    print(result["status"])
    display(pd.DataFrame([{"후보": key, **value["slot_micro"], "Average precision": value["average_precision"], "잠정 일평균 F1": value["daily_station_macro"]["f1"], "실제 결빙일 F1": value["fixed_ice_day_f1"]} for key, value in result["metrics"].items()]))
else:
    print("이 작업공간에는 추정온도 비교가 아직 없습니다.")
audit_path = PRIVATE / "estimated_temperature_forward/audit.json"
if audit_path.exists():
    print(json.loads(audit_path.read_text(encoding="utf-8")))
print("반복 개발 비교입니다. 이미 연 보호3지점의 최종 평가를 대체하지 않습니다.")

## 공식 SFS 의미를 반영한 상태 입력V2

V1은12/13 누적 비율을deicing으로 묶었다. V2는12 chemical wet과13 critical wet을 따로 요약한다. 공식04 양성 정답은 유지하며 V1 결과는 재현용으로 보존한다.

In [ ]:
path = PRIVATE / "surface_state_v2_forward/summary.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    print(result["status"])
    display(pd.DataFrame([{"후보": key, **value["slot_micro"], "Average precision": value["average_precision"], "잠정 일평균 F1": value["daily_station_macro"]["f1"], "실제 결빙일 F1": value["fixed_ice_day_f1"]} for key, value in result["metrics"].items()]))
else:
    print("이 작업공간에는 해당 개발 비교가 아직 없습니다.")
audit_path = PRIVATE / "surface_state_v2_forward/audit.json"
if audit_path.exists():
    print(json.loads(audit_path.read_text(encoding="utf-8")))
print("반복 개발 결과이며 새 독립 최종 성능이 아닙니다.")

## 수정 상태 모델과 열·수분의 고정 평균

두 후보의0.5 산술·기하 평균을 고정하고 첫 겨울 내부OOF에서만 임계값을 결정한다. 추가 보호 평가·가중치 튜닝은 하지 않는다.

In [ ]:
path = PRIVATE / "surface_state_v2_ensemble/summary.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    print(result["status"])
    display(pd.DataFrame([{"후보": key, **value["slot_micro"], "Average precision": value["average_precision"], "잠정 일평균 F1": value["daily_station_macro"]["f1"], "실제 결빙일 F1": value["fixed_ice_day_f1"]} for key, value in result["metrics"].items()]))
else:
    print("이 작업공간에는 해당 개발 비교가 아직 없습니다.")
audit_path = PRIVATE / "surface_state_v2_ensemble/audit.json"
if audit_path.exists():
    print(json.loads(audit_path.read_text(encoding="utf-8")))
print("반복 개발 결과이며 새 독립 최종 성능이 아닙니다.")

## 추가 공식 관측자료의 확보 조건

기상청 API허브 관측지점 PDF에서 대회 사용 지점·보호 지점을 제외한다. 노선 원문을 보존하고 통영-대전선/통영-대전중부 표기만 같은 대상 계열로 대응한다. 지점 ID가 다르더라도 같은 물리 위치이면 독립성이 약하므로1km 이내 중복 위치 후보를 제외한다. 기존 참조 거리 범위를 넘는 점도 별도로 표시한다. 이는 관측 확보·역사 시점·독립 검증의 완료를 의미하지 않는다. 2026-10-09 API 승인을 확인했고 첫 겨울 개발 관측만 수집 중이다. 다음 겨울은 예측·모델 고정 후 한 번 평가할 자료로 분리한다.

In [ ]:
path = PRIVATE / "external/kma_rwis_reference/catalog_summary.json"
if path.exists():
    print(json.loads(path.read_text(encoding="utf-8")))
    display(pd.read_csv(path.parent / "independent_station_candidates.csv", dtype={"station": str}))
else:
    print("이 작업공간에는 공식 추가 지점 후보 검토가 아직 없습니다.")
print("인증키는 노트북·로그에 기록하지 않습니다. 관측 정답이 확보되면 새 평가의 고정·예측 선행 절차가 필요합니다.")

## 외부 첫 겨울 개발 관측과 다음 겨울 검증 분리

9개 후보를 서천·백양·금산·서상4개 공간 그룹으로 묶었다. 그룹 전체를 학습·내부 선택·동적 참조에서 제외하고 다음 겨울을 검증한다. 여기서는 수집된 첫 겨울 관측의 원문 재파싱, 정확한10분 추출, 결측·좌표·고도를 검토한다. 수집 중인 부분 자료는 입력 연결 확인에만 쓰며 모델 학습·선정에는 쓰지 않는다. 다음 겨울 외부 정답은 아직 열지 않았다.

In [ ]:
path = PRIVATE / "external/kma_rwis_development_v1/summary.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    print({key: result[key] for key in ["status", "downloaded_windows_audited", "expected_windows_full_winter", "exact_ten_minute_rows", "valid_state_rows", "ice_rows", "reserved_validation_opened"]})
    display(pd.DataFrame(result["stations"])[["station", "route", "spatial_group", "ten_minute_rows", "valid_state_rows", "ice_rows", "maximum_catalog_coordinate_difference_m"]])
else:
    print("이 작업공간에는 외부 개발 관측의 감사 결과가 아직 없습니다.")
print("추가 개발 자료 확보는 모델 개선의 증명이 아닙니다. 다음 겨울은 예측 고정 전 수집하지 않습니다.")

## 추가 관측의 편중과 상태·온도 관계

특정 지점의 Ice 집중과 온도 관계를 정답 수정 없이 진단한다. 전체 첫 겨울 대회 개발 자료와 추가자료의 같은 시각집합을 구분해 비교한다. 지점과 기간이 달라 센서 오류나 모델 개선을 확정하지 않는다. 연속 관측 슬롯은 실제 결빙 지속시간이 아니다.


In [ ]:
path = PRIVATE / "external_quality/latest.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    snapshot = ROOT / result["snapshot"]
    import hashlib
    assert hashlib.sha256((snapshot / "summary.json").read_bytes()).hexdigest() == result["summary_sha256"]
    print({key: result[key] for key in ["status", "external_rows", "external_ice_rows", "largest_station_ice_share", "external_ice_temperature_above_2c_rows", "competition_matching_external_times"]})
    display(pd.read_csv(snapshot / "external_station_quality.csv", dtype={"station": str}))
    from IPython.display import Image
    display(Image(filename=str(snapshot / "quality_overview.png")))
else:
    print("이 작업공간에는 추가 관측의 품질 진단 스냅샷이 아직 없습니다.")
print("관측04 정답은 그대로 유지하며, 이 진단은 독립 성능 평가가 아닙니다.")


## 외부 첫 겨울의 정답 없는 입력 선계산

추가9지점의 정적 좌표와 허용된14개 개발 지점의 과거 관측만으로 전체10분 입력을 생성한다. 원천·소스 서명을 확인하고 실제 감사 시각에서 직접 계산과 비교한다. 외부 대상 상태·온도 정답은 이 캐시에 포함하지 않는다. 전체 수집 완료 전의 정답 결합 스냅샷은 학습에 사용하지 않는다.

In [ ]:
path = PRIVATE / "external_development_inputs_v1/manifest.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    print({key: result[key] for key in ["status", "first_time_kst", "last_time_kst", "allowed_dynamic_donors", "target_observations_used", "frozen_files_unchanged"]})
    rows = []
    import hashlib
    for station, record in result["records"].items():
        assert hashlib.sha256((path.parent / f"{station}.parquet").read_bytes()).hexdigest() == record["sha256"]
        rows.append({"station": station, "rows": record["rows"]})
    display(pd.DataFrame(rows))
    parity = path.parent / "parity_verification_20261009.json"
    if parity.exists():
        evidence = json.loads(parity.read_text(encoding="utf-8"))
        assert hashlib.sha256(path.read_bytes()).hexdigest() == evidence["cache_manifest_sha256"]
        print(evidence)
else:
    print("이 작업공간에는 외부 개발 입력 캐시가 아직 없습니다.")
print("전체 입력 생성은 모델 성능 개선을 의미하지 않습니다. 외부 다음 겨울 정답은 예측 고정 전 열지 않습니다.")


## 공식 2024 도로망과 제공 구간의 공간 비교

국가교통정보센터의 2024-03-25 배포본으로 방향별 링크·연결로·사용 여부를 검토한다. 노선번호만으로는 호남선과 통영대전선 밖의 도로가 섞이므로 공식 노선명을 함께 제한했다. 원본 WKT를 보존해 변환했으며 datum 변환 정확도는 미확정이다. 25m 간격의 최근접 샘플은 근접도 진단이고 본선 소속·상하행·공식 이정 확정이 아니다. 기존467개 구간과 모델·제출 초안은 유지한다.

In [ ]:
path = PRIVATE / "spatial/official_nodelink_review_v1/summary.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    import hashlib
    for name, digest in result["output_sha256"].items():
        assert hashlib.sha256((path.parent / name).read_bytes()).hexdigest() == digest
    assert hashlib.sha256((path.parent / "audit_source_snapshot.py").read_bytes()).hexdigest() == result["source_sha256"]["review_script"]
    print({key: result[key] for key in ["status", "source_publication_date", "national_named_route_links", "orientation_checks", "official_chainage_verified", "final_submission_use_rights_confirmed"]})
    display(pd.DataFrame(result["routes"])[["route", "provided_segments", "approx_length_fraction_within_30m_active", "approx_length_fraction_within_30m_non_ramp", "approx_length_fraction_nearest_ramp"]])
    print(json.loads((path.parent / "projection_validation.json").read_text(encoding="utf-8")))
    from IPython.display import Image
    display(Image(filename=str(path.parent / "diagnostics/non_ramp_coverage_comparison.png")))
else:
    print("이 작업공간에는 공식 2024 도로망 공간 감사 결과가 아직 없습니다.")
print("최근접 연결로 후보 비율은 실제 소속의 확정값이 아닙니다. 최종 구간·이정·자료 이용 조건은 별도 검토가 필요합니다.")

## 공식 방향별 500m·1km 구간 후보와 검토 대상

공식 권역 코드와 노선명·사용·비연결로를 함께 제한하고, 공유 노드와 진행 방향의 연속성을 검토했다. 여섯 경로는 명명된 공식 끝점 사이의 후보다. 서해안선 한 접속부의 원본6.39m 틈과 방향 민감도 실패를 보존하며 가상 연결선을 추가하지 않는다. 행정권역 코드만으로 전북 경계를 확정하지 않으며, SGIS2024 시도경계가 도착하면 별도 버전으로 잘라낸다. 로컬 거리 원점은 공식 이정이 아니고 위험점수는 모두 미산정이다.

In [ ]:
path = PRIVATE / "spatial/official_directed_segment_candidates_v1/summary.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    import hashlib
    for name, digest in result["outputs_sha256"].items():
        assert hashlib.sha256((path.parent / name).read_bytes()).hexdigest() == digest
    print({key: result[key] for key in ["status", "selected_path_links", "sizes_m", "angular_sensitivity_checks_passed", "angular_sensitivity_checks_total", "all_angular_choices_stable", "final_geometry_approved", "province_boundary_clip_verified", "risk_scored"]})
    display(pd.DataFrame(result["directional_paths"])[["route", "flow_label", "links", "geometric_length_m", "maximum_source_join_gap_m"]])
    display(pd.read_csv(path.parent / "angle_tangent_sensitivity.csv"))
    audit = json.loads((path.parent / "independent_lineage_verification.json").read_text(encoding="utf-8"))
    assert hashlib.sha256(path.read_bytes()).hexdigest() == audit["summary_sha256"]
    print(audit)
    from IPython.display import Image
    display(Image(filename=str(path.parent / "source_gap_diagnostic.png")))
else:
    print("이 작업공간에는 공식 방향별 구간 후보가 아직 없습니다.")
print("경계·이용 조건·서해안 접속부 검토가 남았습니다. 기존 구간과 위험도 결과를 대체하지 않습니다.")

## 외부 첫 겨울 중첩 검증과 정답 전 예측 고정

합산 지표와 지점별 상충을 함께 확인합니다. 이 셀은 관측 수집·학습·평가를 실행하지 않습니다.


In [ ]:
audit_path = PRIVATE / "external_nested_audit_v1/summary.json"
if audit_path.exists():
    result = json.loads(audit_path.read_text(encoding="utf-8"))
    print({key: result[key] for key in ["status", "prediction_rows", "fit_receipts_checked", "identical_fit_reuse_receipts_checked", "unique_thresholds_recalculated", "reference_records_checked", "classification_slot_micro", "temperature_paired", "group_choices"]})
    diagnostics = json.loads((audit_path.parent / "paired_diagnostics.json").read_text(encoding="utf-8"))
    display(pd.DataFrame([{"station": station, "baseline_mae": value["temperature"]["baseline"]["mae"], "selected_mae": value["temperature"]["candidate"]["mae"], "tp": value["selected"]["slot_micro"]["tp"], "fp": value["selected"]["slot_micro"]["fp"], "fn": value["selected"]["slot_micro"]["fn"]} for station, value in diagnostics["station"].items()]))
else:
    print("이 작업공간에는 외부 중첩 검증의 독립 재검산 결과가 아직 없습니다.")
receipt_path = PRIVATE / "external_reserved_predictions_v1/predictions_receipt.json"
if receipt_path.exists():
    receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
    print({key: receipt[key] for key in ["status", "rows", "sealed_at_utc", "reserved_outcomes_opened_at_seal"]})
    print("예측 고정 파일 수:", len(receipt["files"]))
print("첫 겨울 비교는 개발 근거입니다. 다음 겨울 정답 수집 이후 모델·임계값·후보 선택을 변경하지 않습니다.")


## 방향별 후보의 공식 도로환경 속성

원본 속성 대조·유형별 길이 보존·CSV/GPKG 재읽기 결과를 확인합니다.


In [ ]:
path = PRIVATE / "spatial/official_directed_environment_review_v1/summary.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    print({key: result[key] for key in ["status", "original_dbf_rows_scanned", "selected_original_links_exactly_verified", "original_link_type_counts", "sizes", "final_geometry_approved", "physical_bridge_tunnel_extent_verified", "risk_scored", "reserved_frozen_files_unchanged"]})
    import hashlib
    for name, digest in result["files_sha256"].items():
        assert hashlib.sha256((path.parent / name).read_bytes()).hexdigest() == digest
    display(pd.read_csv(path.parent / "route_structure_code_lengths.csv"))
    plot_receipt = json.loads((path.parent / "plot_review_verification.json").read_text(encoding="utf-8"))
    assert hashlib.sha256(path.read_bytes()).hexdigest() == plot_receipt["environment_summary_sha256"]
    plot = path.parent / plot_receipt["plot_file"]
    assert hashlib.sha256(plot.read_bytes()).hexdigest() == plot_receipt["plot_sha256"]
    from IPython.display import Image
    display(Image(filename=str(plot)))
else:
    print("이 작업공간에는 방향별 후보의 공식 도로환경 속성 검토가 아직 없습니다.")
print("이 속성은 과거 원본 링크 코드입니다. 실제 구조물 경계·전북 범위·현재 속도·결빙 위험등급은 확인 전입니다.")


## 시설 폴리곤 중첩과 원본 코드 불일치

수평 중첩 민감도·원본 노드명·영향 조각을 읽습니다. 원본 코드와 고정 모델은 유지합니다.

In [ ]:
path = PRIVATE / "spatial/official_directed_facility_overlap_review_v1/summary.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    import hashlib
    for name, digest in result["files_sha256"].items():
        assert hashlib.sha256((path.parent / name).read_bytes()).hexdigest() == digest
    print({key: result[key] for key in ["status", "selected_links", "overlap_partition_conservation_checks", "all_partition_overlap_lengths_conserved", "physical_carriageway_membership_verified", "risk_scored"]})
    display(pd.read_csv(path.parent / "structure_code_polygon_comparison.csv", dtype={"official_road_type": str}))
    diagnostic = path.parent / "diagnostic_review_v1"
    cases = json.loads((diagnostic / "summary.json").read_text(encoding="utf-8"))
    assert hashlib.sha256(path.read_bytes()).hexdigest() == cases["input_overlap_summary_sha256"]
    for name, digest in cases["files_sha256"].items():
        assert hashlib.sha256((diagnostic / name).read_bytes()).hexdigest() == digest
    display(pd.read_csv(diagnostic / "source_conflict_cases.csv", dtype={"LINK_ID": str, "ROAD_TYPE_preserved": str}))
    print("영향 구간 수:", cases["affected_unique_segments_by_size"])
    plot_receipt = json.loads((diagnostic / "plot_review_verification.json").read_text(encoding="utf-8"))
    assert hashlib.sha256((diagnostic / "summary.json").read_bytes()).hexdigest() == plot_receipt["diagnostic_summary_sha256"]
    assert hashlib.sha256((diagnostic / plot_receipt["plot_file"]).read_bytes()).hexdigest() == plot_receipt["plot_sha256"]
    assert hashlib.sha256((diagnostic / plot_receipt["renderer_file"]).read_bytes()).hexdigest() == plot_receipt["renderer_sha256"]
    from IPython.display import Image
    display(Image(filename=str(diagnostic / "horizontal_overlap_sensitivity.png")))
    display(Image(filename=str(diagnostic / plot_receipt["plot_file"])))
else:
    print("이 작업 공간에는 시설 중첩 검토 결과가 아직 없습니다.")
print("수평 중첩은 실제 차로·층위·자료 시점·결빙 위험의 확인 결과가 아닙니다. 원본 코드와 고정 모델을 유지합니다.")


## 제공 설명서 기반 도로교 구분

제공 설명서의 종류 코드로 도로교만의 수평 중첩과 제외 영향을 확인합니다.

In [ ]:
path = PRIVATE / "spatial/official_directed_facility_overlap_review_v1/road_bridge_kind_review_v1/summary.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    import hashlib
    for name, digest in result["files_sha256"].items():
        assert hashlib.sha256((path.parent / name).read_bytes()).hexdigest() == digest
    assert hashlib.sha256((path.parent.parent / "summary.json").read_bytes()).hexdigest() == result["input_overlap_summary_sha256"]
    print({key: result[key] for key in ["status", "manual_version", "manual_authored_date", "kind_counts", "material_counts", "overlap_partition_conservation_checks", "segments_affected_by_nonroad_filter_30m", "polygon_vintage_for_observation_winters_verified", "risk_scored"]})
    display(pd.read_csv(path.parent / "road_bridge_structure_code_comparison.csv", dtype={"ROAD_TYPE": str}))
    display(pd.read_csv(path.parent / "links_affected_by_nonroad_bridge_filter.csv", dtype={"LINK_ID": str, "ROAD_TYPE": str}))
else:
    print("이 작업 공간에는 도로교 종류 검토가 아직 없습니다.")
print("설명서 작성일·XML 처리일·FMTA 일련번호는 실제 시설 관측일이 아닙니다. 종류 확인과 실제 차로 소속을 구분합니다.")


## 도로교 장축·도로 국지 방향의 민감도

분류 설정 변화와 배타 길이 검사를 읽습니다. 물리적 차로 소속·상하 교차 판별은 확인 전입니다.

In [ ]:
path = PRIVATE / "spatial/official_directed_facility_overlap_review_v1/bridge_direction_sensitivity_v1/summary.json"
if path.exists():
    result = json.loads(path.read_text(encoding="utf-8"))
    import hashlib
    for name, digest in result["files_sha256"].items():
        assert hashlib.sha256((path.parent / name).read_bytes()).hexdigest() == digest
    assert hashlib.sha256((path.parent.parent / "road_bridge_kind_review_v1/summary.json").read_bytes()).hexdigest() == result["input_kind_summary_sha256"]
    print({key: result[key] for key in ["status", "road_bridge_polygons_axis_ratio_lt_2", "bridge_link_pairs", "sensitivity_settings", "pair_stable_class_counts", "exclusive_class_partition_and_prior_overlap_checks", "physical_carriageway_or_vertical_membership_verified", "risk_scored"]})
    display(pd.read_csv(path.parent / "direction_class_sensitivity_summary.csv"))
    from IPython.display import Image
    display(Image(filename=str(path.parent / "bridge_direction_sensitivity.png")))
else:
    print("이 작업 공간에는 교량 방향 민감도 결과가 아직 없습니다.")
print("조합별 길이를 합산하지 않습니다. 검토 설정의 분류는 실제 교량 차로·상하 교차·결빙 위험의 정답이 아닙니다.")
